# Reproducing the figures

Code for every figure in the paper on CGM cooling under shock, adiabatic, and continuous compression.

The physics functions live in four modules, each named after the notebook it came from:

| Module | Contents |
|---|---|
| `cooling.py` | constants, `Lambda_z0`, `tcool_P` |
| `shock_transition.py` | `shock_tran`, `time_diff` (shock) |
| `adiabatic_process.py` | `shock_tran`, `adi_tran`, `time_diff` (adiabatic), `kai_func` |
| `infall_galaxy.py` | `cont_radi`, `cool_radiation`, `entropy_diff`, `cool_radiation_single` |

The two studies both define `shock_tran` and `time_diff`, so they are imported as `shock.` and `adi.`.

**Requirement:** `data/z_0.000.hdf5`, the z = 0 cooling table of Wiersma et al. (2009).

## Setup

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
from matplotlib import cm
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
from scipy.interpolate import interp1d
from scipy.optimize import root_scalar

from cooling import mu, mH, kb, gamma, tcool_P
import shock_transition as shock
import adiabatic_process as adi
from infall_galaxy import cont_radi, cool_radiation, entropy_diff, cool_radiation_single

scale = 6 / 2.5
plt.rcParams.update({
    "axes.titlesize":  9 * scale,
    "axes.labelsize":  9 * scale,
    "xtick.labelsize": 8 * scale,
    "ytick.labelsize": 8 * scale,
    "legend.fontsize": 8 * scale,
})

## Figure 1 — entropy proxy and constant cooling-time curves
Z = 0.3 Z☉. White arrows show an adiabatic jump (χ = 10); black arrows show a shock (M = 5).

In [ ]:
n2_adi, T2_adi = adi.adi_tran(1e-3, 10**4.5, 10)
n2_adi_arrow = (T2_adi-10**4.5, n2_adi-1e-3)
n2_adi_2, T2_adi_2 = adi.adi_tran(10**(-3.99) , 10**(5.22) , 10)
n2_adi_arrow_2= (T2_adi_2 -10**(5.22), n2_adi_2 -10**(-3.99) )
u2_shock_2, n2_shock_2, T2_shock_2 = adi.shock_tran(5, 10**(-3.99),10**(5.22))
u2_shock, n2_shock, T2_shock = adi.shock_tran(5, 1e-3, 10**4.5)
n2_shock_arrow = ((T2_shock-1) * 10**4.5, (n2_shock - 1) * 1e-3)
n2_shock_arrow_2 = ((T2_shock_2 -1) * 10**(5.22) , (n2_shock_2 - 1) * 10**(-3.99) )

In [ ]:
"""
plot of tcool = t_H varying n
"""
T_range = np.linspace(4, 7, 5000)
n_range = np.linspace(-6, 0, 5000)

metallicity = 0.3
t_H = 1e17
log_tH = np.log10(t_H)

pairs_n = []

for ti in T_range:
    tcool_vals = tcool_P(10**ti, 10**n_range, metallicity)

    mask = (
        (tcool_vals - log_tH >= np.log10(0.999)) &
        (tcool_vals - log_tH < np.log10(1.001))
    )

    n_col = n_range[mask]

    if n_col.size > 0:
        pairs_n.extend([(ti, ni) for ni in n_col])
T_n_H = [p[0] for p in pairs_n]
n_H = [p[1] for p in pairs_n]
bottom_func = interp1d(T_n_H, n_H, kind = 'cubic', fill_value="extrapolate")

t_range = np.array((1e16, 10**(16.5), 1e14, 1e15))
T_range = np.linspace(4, 7, 2000)
data = {}
for ti in t_range:
    data[ti] = []
    for Ti in T_range:
        def tcool_midel(x):
          return tcool_P(10**Ti, 10**x, metallicity) - np.log10(ti)
        if tcool_midel(bottom_func(Ti))*tcool_midel(0) <= 0:
          solution = root_scalar(tcool_midel, bracket = [bottom_func(Ti), 0], method='brentq')
          data[ti].append((Ti, solution.root))
T0, n_0 = zip(*data[1e16])
T1, n_1 = zip(*data[10**(16.5)])
T2, n_2 = zip(*data[1e14])
T3, n_3 = zip(*data[1e15])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
n1_0 = 1e-3
T1_0 = 10**4.5
plt.rcParams.update({"legend.fontsize": 6 * scale})
n2_0 = 10**(-3.99)
T2_0 = 10**(5.22)
n_range = 10** np.linspace(-6, 0, 1000)
T_range = 10** np.linspace(4, 7, 1000)
T , N= np.meshgrid(T_range, n_range)
Z = adi.kai_func(T, N)
bounds = [9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24 ]
cmap = plt.get_cmap('rainbow', len(bounds)-1)
norm = colors.BoundaryNorm(bounds, cmap.N)
# Plot heatmap
plt.pcolormesh(T, N, Z, cmap= cmap, norm = norm)
plt.plot(10**np.array(T2), 10**np.array(n_2), "black", linestyle=(0, (5, 2, 5, 2, 1, 2)), label = r"$t_{cool} = 10^{14}$")
plt.plot(10**np.array(T3), 10**np.array(n_3), "black", linestyle=(0, (5, 2, 5, 2, 5, 2, 1, 2)), label = r"$t_{cool} = 10^{15}$")
plt.plot(10**np.array(T0), 10**np.array(n_0), "black", linestyle = "-." , label = r"$t_{cool} = 10^{16}$")
plt.plot(10**np.array(T_n_H), 10**np.array(n_H), "black", linestyle = "--", label = "$t_{cool} = 10^{17}$")
plt.colorbar(label=r"$log_{10}(K)$")
aid_arrow0 = n2_adi_arrow[0]
aid_arrow1 = n2_adi_arrow[1]
shock_arrow0 = n2_shock_arrow[0]
shock_arrow1 = n2_shock_arrow[1]
aid_arrow2 = n2_adi_arrow_2[0]
aid_arrow3 = n2_adi_arrow_2[1]
shock_arrow2 = n2_shock_arrow_2[0]
shock_arrow3 = n2_shock_arrow_2[1]
ax.annotate('', xy=(T1_0 + aid_arrow0, n1_0 + aid_arrow1), xytext=(T1_0, n1_0),
            arrowprops=dict(facecolor='white', edgecolor='white', width=1, headwidth=4, headlength=3))
ax.annotate('', xy=(T1_0 + shock_arrow0, n1_0 + shock_arrow1), xytext=(T1_0, n1_0),
            arrowprops=dict(facecolor='black', edgecolor='black', width=1, headwidth=4, headlength=3))
ax.annotate('', xy=(T2_0 + aid_arrow2, n2_0 + aid_arrow3), xytext=(T2_0, n2_0),
            arrowprops=dict(facecolor='white', edgecolor='white', width=1, headwidth=4, headlength=3))
ax.annotate('', xy=(T2_0 + shock_arrow2, n2_0 + shock_arrow3), xytext=(T2_0, n2_0),
            arrowprops=dict(facecolor='black', edgecolor='black', width=1, headwidth=4, headlength=3))
adi_proxy = Line2D([0], [0], color='white', lw=2, label='adi arrow',  markeredgecolor='black')
shock_proxy = Line2D([0], [0], color='black', lw=2, label='shock arrow')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(1e-6, 1)
ax.set_xlim(1e4, 1e7)
ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax.tick_params(axis='both', which='major', length=6, width=1)
ax.tick_params(axis='both', which='minor', length=3, width=0.8)
ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs='auto', numticks=100))
ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs='auto', numticks=100))
for spine in ax.spines.values():
    spine.set_linewidth(1)
ax.set_xlabel('Temperature [K]')
ax.set_ylabel(r'Density [$cm^{-3}$]')
plt.tight_layout()
handles, labels = ax.get_legend_handles_labels()
handles += [adi_proxy, shock_proxy]
labels += ['adi arrow', 'shock arrow']
plt.legend(loc='lower right')
plt.show()
plt.rcParams.update({"legend.fontsize": 8 * scale})

## Figure 2 — shock transition: log₁₀(t₂/t₁)

### (a) Mach number vs T  (n = 10⁻⁴ cm⁻³, Z = 0.3 Z☉)

In [ ]:
"""Plot of tcool varying March number"""
T_range = np.linspace(4, 7, 5000)
n = 1e-4
metallicity = 0.3
data = {}
time_range = np.array((14, 15, 16, 17, 16.5))
for ti_val in time_range:
  pairs_M = []
  tcool_vals = tcool_P(10**T_range, n, metallicity)
  mask = (
        (tcool_vals - ti_val > np.log10(0.98)) &
        (tcool_vals - ti_val < np.log10(1.02))
      )
  t_col = T_range[mask]
  data[ti_val] = []
  if t_col.size > 0:
        pairs_M.extend(list(t_col))
        data[ti_val].append(pairs_M[0])
        data[ti_val].append(pairs_M[-1])
"""Note that the line at 1e17 is close to hubble time, but is not exactly equal to hubble time"""

In [ ]:
M_range = np.linspace(1, 10, 1000)
T_range = np.linspace(4, 7, 1000)
n = 1e-4
metallicity = 0.3
T , M= np.meshgrid(T_range, M_range)
Z = shock.time_diff(M, 10**T, n, mH* (mu), metallicity)
fig = plt.figure(figsize=(6, 4.5))
plt.imshow(
    Z,
    extent=[T_range.min(), T_range.max(), M_range.min(), M_range.max()],
    origin='lower',
    aspect='auto',
    cmap='seismic', # diverging colormap
    vmin=-1,
    vmax=1
)
T_7_0 = data[17][0]
T_7_1 = data[17][1]
T_6_0 = data[16][0]
T_6_1 = data[16][1]
T_65_0 = data[16.5][0]
T_65_1 = data[16.5][1]
plt.axvline(x = T_7_0, color =  "black", linestyle = "--")
plt.axvline(x = T_7_1, color =  "black", linestyle = "--")
plt.axvline(x = T_6_0, color =  "black", linestyle = "-.")
plt.axvline(x = T_6_1, color =  "black", linestyle = "-." )
plt.axvline(x = T_65_0, color =  "black", linestyle=(0, (5, 2, 5, 2, 1, 2)))
plt.axvline(x = T_65_1, color =  "black", linestyle=(0, (5, 2, 5, 2, 1, 2)))
plt.xlabel(r"$log_{10}(\mathrm{T}/\mathrm{K})$")
plt.ylabel(r"$\mathrm{Mach \ Number}$")
ax = plt.gca()
ax.xaxis.set_minor_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(1))
plt.show()

### (b) density vs T  (M = 5, Z = 0.3 Z☉)

In [ ]:
"""
plot of tcool = t_H varying n
"""
T_range = np.linspace(4, 7, 5000)
n_range = np.linspace(-6, 0, 5000)
metallicity = 0.3
t_H = 4.4e17
log_tH = np.log10(t_H)
pairs_n = []
for ti in T_range:
    tcool_vals = tcool_P(10**ti, 10**n_range, metallicity)
    mask = (
        (tcool_vals - log_tH >= np.log10(0.999)) &
        (tcool_vals - log_tH < np.log10(1.001))
    )
    n_col = n_range[mask]
    if n_col.size > 0:
        pairs_n.extend([(ti, ni) for ni in n_col])
T_n_H = [p[0] for p in pairs_n]
n_H = [p[1] for p in pairs_n]
bottom_func = interp1d(T_n_H, n_H, kind = 'cubic', fill_value="extrapolate")

t_range = np.array((1e16, 10**(16.5)))
T_range = np.linspace(4, 7, 2000)
data = {}
for ti in t_range:
    data[ti] = []
    for Ti in T_range:
        def tcool_midel(x):
          return tcool_P(10**Ti, 10**x, metallicity) - np.log10(ti)
        if tcool_midel(bottom_func(Ti))*tcool_midel(0) <= 0:
          solution = root_scalar(tcool_midel, bracket = [bottom_func(Ti), 0], method='brentq')
          data[ti].append((Ti, solution.root))
T0, n_0 = zip(*data[1e16])
T2, n_2 = zip(*data[10**(16.5)])

In [ ]:
n_range = np.linspace(-6, 0, 1000)
T_range = np.linspace(4, 7, 1000)
M_1 = 5
metallicity = 0.3
T , N= np.meshgrid(T_range, 10**n_range)
Z = shock.time_diff(M_1, 10**T, N, mH* (mu), metallicity)
fig = plt.figure(figsize=(6, 4.5))
plt.imshow(
    Z,
    extent=[T_range.min(), T_range.max(), n_range.min(), n_range.max()],
    origin='lower',
    aspect='auto',
    cmap= 'seismic', # diverging colormap
    vmin=-1,
    vmax=1
)
plt.plot(T_n_H, n_H, "black", linestyle = "--")
plt.plot(T0, n_0, "black", linestyle = "-.")
plt.plot(T2, n_2, "black", linestyle=(0, (5, 2, 5, 2, 1, 2)))
plt.xlabel(r"$log_{10}(\mathrm{T}/\mathrm{K})$")
plt.ylabel(r"$log_{10}(\mathrm{n}/\mathrm{cm}^{-3})$")
ax = plt.gca()
ax.xaxis.set_minor_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(1))
plt.show()

### (c) metallicity vs T  (M = 5, n = 10⁻⁴ cm⁻³)

In [ ]:
'''
Plot of tcool = t_H varying metallicity
'''
T_range = np.linspace(4, 7, 5000)
m_range = np.linspace(-2, 0, 5000)
n = 1e-4
t_H = 1e17
log_tH = np.log10(t_H)
pairs_m = []
for ti in T_range:
    tcool_vals = tcool_P(10**ti, n, 10**m_range)
    mask = (
        (tcool_vals - log_tH >= np.log10(0.999)) &
        (tcool_vals - log_tH <= np.log10(1.001))
    )
    m_col = m_range[mask]
    if m_col.size > 0:
        pairs_m.extend([(ti, Mi) for Mi in m_col])
T_m = [p[0] for p in pairs_m]
m = [10**p[1] for p in pairs_m]

T_m_arr = np.array(T_m)
m_arr = np.array(m)
combined_data = np.column_stack((T_m_arr, m_arr))
sorted_combined_data = combined_data[combined_data[:, 0].argsort()]
unique_T_m_arr, unique_indices = np.unique(sorted_combined_data[:, 0], return_index=True)
unique_m_arr = sorted_combined_data[unique_indices, 1]
bottom_func_2 = interp1d(unique_T_m_arr, unique_m_arr, kind='linear', fill_value="extrapolate")

t_range = np.array((10**(16), 10**(16.5)))
T_range = np.linspace(4, 7, 2000)
data = {}
for ti in t_range:
    data[ti] = []
    for Ti in T_range:
        def tcool_midel(x):
          return tcool_P(10**Ti, n, x) - np.log10(ti)
        if tcool_midel(bottom_func_2(Ti))*tcool_midel(1) <= 0:
          solution = root_scalar(tcool_midel, bracket = [bottom_func_2(Ti), 1], method='brentq')
          data[ti].append((Ti, solution.root))
T0, m_0 = zip(*data[10**(16)])
T1, m_1 = zip(*data[10**(16.5)])

In [ ]:
Metal_range = np.linspace(0.01, 1, 1000)
T_range = np.linspace(4, 7, 1000)
M_1 = 5
n = 1e-4
T , Metallicity= np.meshgrid(T_range, Metal_range)
Z = shock.time_diff(M_1, 10**T, n, mH* (mu), Metallicity)
fig = plt.figure(figsize=(6, 4.5))
plt.imshow(
    Z,
    extent=[T_range.min(), T_range.max(), Metal_range.min(), Metal_range.max()],
    origin='lower',
    aspect='auto',
    cmap='seismic', # diverging colormap
    vmin=-1,
    vmax=1
)
plt.plot(T_m_arr, m_arr, "black", linestyle = "--", label = r"$t_{cool} = 10^{17}$")
plt.plot(T0, m_0, "black", linestyle = "-." , label = r"$t_{cool} = 10^{16}$")
plt.plot(T1, m_1, "black", linestyle=(0, (5, 2, 5, 2, 1, 2)), label = r"$t_{cool} = 3 \times 10^{16}$")
plt.xlabel(r"$log_{10}(\mathrm{T}/\mathrm{K})$")
plt.ylabel(r"$\mathrm{Metallicity}$/$Z_{\odot}$")
ax = plt.gca()
ax.xaxis.set_minor_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(1))
plt.legend(fontsize=16, loc='lower right')
plt.show()

In [ ]:
# colour bar for Figures 2 and 3
fig, cax = plt.subplots(figsize=(.5, 4.5), constrained_layout=True)
norm = mpl.colors.Normalize(vmin=-1, vmax=1)
sm = mpl.cm.ScalarMappable(norm=norm, cmap='seismic')
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label(r'$\log_{10}(t_2/t_1)$')
plt.show()

## Figure 3 — adiabatic transition: log₁₀(t₂/t₁)

### (a) κ vs T  (n = 10⁻⁴ cm⁻³, Z = 0.3 Z☉)

In [ ]:
"""Plot of tcool with varying kai"""
T_range = np.linspace(4, 7, 5000)
n = 1e-4
metallicity = 0.3
data = {}
time_range = np.array((16.5, 16, 17))
for ti_val in time_range:
  pairs_M = []
  tcool_vals = tcool_P(10**T_range, n, metallicity)
  mask = (
        (tcool_vals - ti_val > np.log10(0.99)) &
        (tcool_vals - ti_val < np.log10(1.01))
      )
  t_col = T_range[mask]
  data[ti_val] = []
  if t_col.size > 0:
        pairs_M.extend(list(t_col))
        data[ti_val].append(pairs_M[0])
        data[ti_val].append(pairs_M[-1])

In [ ]:
kai_range = np.linspace(1, 10, 1000)
T_range = np.linspace(4, 7, 1000)
n = 1e-4
metallicity = 0.3
T , Kai= np.meshgrid(T_range, kai_range)
Z = adi.time_diff(n, 10**T, Kai, metallicity)
fig = plt.figure(figsize=(6, 4.5))
plt.imshow(
    Z,
    extent=[T_range.min(), T_range.max(), kai_range.min(), kai_range.max()],
    origin='lower',
    aspect='auto',
    cmap= 'seismic', # diverging colormap
    vmin=-1,
    vmax=1
)
T_7_0 = data[17][0]
T_7_1 = data[17][1]
T_6_0 = data[16][0]
T_6_1 = data[16][1]
T_5_0 = data[16.5][0]
T_5_1 = data[16.5][1]
plt.axvline(x = T_7_0, color =  "black", linestyle = "-")
plt.axvline(x = T_7_1, color =  "black", linestyle = "-")
plt.axvline(x = T_6_0, color =  "black", linestyle = "--")
plt.axvline(x = T_6_1, color =  "black", linestyle = "--" )
plt.axvline(x = T_5_0, color =  "black", linestyle="-.")
plt.axvline(x = T_5_1, color =  "black", linestyle="-.")
plt.xlabel(r"$log_{10}(\mathrm{T/K})$")
plt.ylabel(r"$\kappa$")
ax = plt.gca()
ax.xaxis.set_minor_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))
plt.show()

### (b) density vs T  (κ = 10, Z = 0.3 Z☉)

In [ ]:
"""
plot of tcool = t_H varying n
"""
T_range = np.linspace(4, 7, 5000)
n_range = np.linspace(-6, 0, 5000)
metallicity = 0.3
t_H = 1e17
log_tH = np.log10(t_H)
pairs_n = []
for ti in T_range:
    tcool_vals = tcool_P(10**ti, 10**n_range, metallicity)
    mask = (
        (tcool_vals - log_tH >= np.log10(0.999)) &
        (tcool_vals - log_tH < np.log10(1.001))
    )
    n_col = n_range[mask]
    if n_col.size > 0:
        pairs_n.extend([(ti, ni) for ni in n_col])
T_n_H = [p[0] for p in pairs_n]
n_H = [p[1] for p in pairs_n]
bottom_func = interp1d(T_n_H, n_H, kind = 'cubic', fill_value="extrapolate")

t_range = np.array((1e16, 10**(16.5)))
T_range = np.linspace(4, 7, 2000)
data = {}
for ti in t_range:
    data[ti] = []
    for Ti in T_range:
        def tcool_midel(x):
          return tcool_P(10**Ti, 10**x, metallicity) - np.log10(ti)
        if tcool_midel(bottom_func(Ti))*tcool_midel(0) <= 0:
          solution = root_scalar(tcool_midel, bracket = [bottom_func(Ti), 0], method='brentq')
          data[ti].append((Ti, solution.root))
T0, n_0 = zip(*data[1e16])
T1, n_1 = zip(*data[10**(16.5)])

In [ ]:
n_range = np.linspace(-6, 0, 1000)
T_range = np.linspace(4, 7, 1000)
kai = 10
metallicity = 0.3
T , N= np.meshgrid(T_range, n_range)
Z = adi.time_diff(10**N, 10**T, kai, metallicity)
fig = plt.figure(figsize=(6, 4.5))
plt.imshow(
    Z,
    extent=[T_range.min(), T_range.max(), n_range.min(), n_range.max()],
    origin='lower',
    aspect='auto',
    cmap='seismic', # diverging colormap
    vmin=-1,
    vmax=1
)
plt.plot(T_n_H, n_H, "black", linestyle = "-")
plt.plot(T0, n_0, "black", linestyle = "--" )
plt.plot(T1, n_1, "black", linestyle="-.")
plt.xlabel(r"$log_{10}(\mathrm{T/K})$")
plt.ylabel(r"$log_{10}(\mathrm{n}/\mathrm{cm}^{-3})$")
ax = plt.gca()
ax.xaxis.set_minor_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))
plt.show()

### (c) metallicity vs T  (κ = 10, n = 10⁻⁴ cm⁻³)

In [ ]:
'''
Plot of tcool = t_H varying metallicity
'''
T_range = np.linspace(4, 7, 5000)
m_range = np.linspace(0.01, 1, 5000)
n = 1e-4
t_H = 1e17
log_tH = np.log10(t_H)
pairs_m = []
for ti in T_range:
    tcool_vals = tcool_P(10**ti, n, m_range)
    mask = (
        (tcool_vals - log_tH >= np.log10(0.999)) &
        (tcool_vals - log_tH <= np.log10(1.001))
    )
    m_col = m_range[mask]
    if m_col.size > 0:
        pairs_m.extend([(ti, Mi) for Mi in m_col])
T_m = [p[0] for p in pairs_m]
m = [p[1] for p in pairs_m]

T_m_arr = np.array(T_m)
m_arr = np.array(m)
combined_data = np.column_stack((T_m_arr, m_arr))
sorted_combined_data = combined_data[combined_data[:, 0].argsort()]
unique_T_m_arr, unique_indices = np.unique(sorted_combined_data[:, 0], return_index=True)
unique_m_arr = sorted_combined_data[unique_indices, 1]
bottom_func_2 = interp1d(unique_T_m_arr, unique_m_arr, kind='linear', fill_value="extrapolate")

t_range = np.array((10**(16), 10**(16.5)))
T_range = np.linspace(4, 7, 2000)
data = {}
for ti in t_range:
    data[ti] = []
    for Ti in T_range:
        def tcool_midel(x):
          return tcool_P(10**Ti, n, x) - np.log10(ti)
        if tcool_midel(bottom_func_2(Ti))*tcool_midel(1) <= 0:
          solution = root_scalar(tcool_midel, bracket = [bottom_func_2(Ti), 1], method='brentq')
          data[ti].append((Ti, solution.root))
T0, m_0 = zip(*data[10**(16)])
T1, m_1 = zip(*data[10**(16.5)])

In [ ]:
metal_range = np.linspace(0.01, 1, 1000)
T_range = np.linspace(4, 7, 1000)
kai = 10
n = 1e-4
T , Metallicity= np.meshgrid(T_range, metal_range)
Z = adi.time_diff(n, 10**T, kai, Metallicity)
fig = plt.figure(figsize=(6, 4.5))
plt.imshow(
    Z,
    extent=[T_range.min(), T_range.max(), metal_range.min(), metal_range.max()],
    origin='lower',
    aspect='auto',
    cmap='seismic', # diverging colormap
    vmin=-1,
    vmax=1
)
plt.plot(T_m_arr, m_arr, "black", linestyle = "-", label = r"$t_{cool} = 10^{17}$")
plt.plot(T0, m_0, "black", linestyle = "--" , label = r"$t_{cool} = 10^{16}$")
plt.plot(T1, m_1, "black", linestyle="-.", label = r"$t_{cool} = 10^{16.5}$")
plt.xlabel(r"$log_{10}(\mathrm{T}/ \mathrm{K})$")
plt.ylabel(r"$\mathrm{Metallicity}$/$Z_{\odot}$")
ax = plt.gca()
ax.xaxis.set_minor_locator(MultipleLocator(0.2))
ax.yaxis.set_minor_locator(MultipleLocator(0.1))
plt.legend(fontsize=16, loc ='lower right')
plt.show()

## Figure 4 — single-parcel evolution under continuous compression
T₀ = 10⁵ K, Z = 0.3 Z☉, compression time 3×10¹⁶ s (≈ 1 Gyr).

In [ ]:
n = 8
shades = cm.Blues(np.linspace(0.2, 1.0, n))
cmap = colors.ListedColormap(shades)
bounds = np.arange(-0.5, n)                   # bin edges: -0.5, 0.5, ..., 7.5
norm = colors.BoundaryNorm(bounds, cmap.N)

T0 = 10**(5)
metallicity = 0.3
kai_range = np.array((1, 3, 7, 10, 15, 20, 25, 30))
kai_range_list = list(kai_range)
period = 3* 10**(16)

In [ ]:
"""
Initial condition:
n = 10**(-5.5), T = 1e5, metallicity = 0.3, period = 3e16
"""
n0 = 10**(-5.5)
data_sin2 = {}
fig, ax = plt.subplots()
ax.set_xlim(3.9, 5.7)
ax.set_ylim(-5.7, -1.7)
ax.xaxis.set_minor_locator(MultipleLocator(0.1))
ax.yaxis.set_minor_locator(MultipleLocator(0.2))
for kai_i in kai_range:
    ind = kai_range_list.index(kai_i)
    x_tmp, y_tmp = cool_radiation_single(n0, T0, kai_i, metallicity, period) # log10(n) and log10(T) at each time
    data_sin2[f"{kai_i}n"] = x_tmp[0]
    data_sin2[f"{kai_i}T"] = y_tmp[0]
    ax.plot(data_sin2[f"{kai_i}T"], data_sin2[f"{kai_i}n"], ls='--', lw=2, color=shades[ind])
ax.set_xlabel(r"$\log_{10}(\mathrm{T/K})$")
ax.set_ylabel(r"$\log_{10}(\mathrm{n/cm^{-3}})$")
plt.show()

In [ ]:
"""
Initial condition:
n = 10**(-5), T = 1e5, metallicity = 0.3, period = 3e16
"""
n0 = 10**(-5)
data_sin = {}
fig, ax = plt.subplots()
ax.set_xlim(3.9, 5.7)
ax.set_ylim(-5.7, -1.7)
ax.xaxis.set_minor_locator(MultipleLocator(0.1))
ax.yaxis.set_minor_locator(MultipleLocator(0.2))
for kai_i in kai_range:
    ind = kai_range_list.index(kai_i)
    x_tmp, y_tmp = cool_radiation_single(n0, T0, kai_i, metallicity, period) # log10(n) and log10(T) at each time
    data_sin[f"{kai_i}n"] = x_tmp[0]
    data_sin[f"{kai_i}T"] = y_tmp[0]
    ax.plot(data_sin[f"{kai_i}T"], data_sin[f"{kai_i}n"], ls='--', lw=2, color=shades[ind])
ax.set_xlabel(r"$\log_{10}(\mathrm{T/K})$")
ax.set_ylabel(r"$\log_{10}(\mathrm{n/cm^{-3}})$")
plt.show()

In [ ]:
"""
Initial condition:
n = 10**(-4.5), T = 1e5, metallicity = 0.3, period = 3e16
"""
n0 = 10**(-4.5)
data_sin3 = {}
fig, ax = plt.subplots()
ax.set_xlim(3.9, 5.7)
ax.set_ylim(-5.7, -1.7)
ax.xaxis.set_minor_locator(MultipleLocator(0.1))
ax.yaxis.set_minor_locator(MultipleLocator(0.2))
for kai_i in kai_range:
    ind = kai_range_list.index(kai_i)
    x_tmp, y_tmp = cool_radiation_single(n0, T0, kai_i, metallicity, period) # log10(n) and log10(T) at each time
    data_sin3[f"{kai_i}n"] = x_tmp[0]
    data_sin3[f"{kai_i}T"] = y_tmp[0]
    ax.plot(data_sin3[f"{kai_i}T"], data_sin3[f"{kai_i}n"], ls='--', lw=2, color=shades[ind])
cax = ax.inset_axes([0.4, 0.08, 0.5, 0.04])
cb = ColorbarBase(cax, cmap=cmap, norm=norm,
                  orientation='horizontal',
                  ticks=np.arange(0,n, 2 ))
cax.xaxis.set_ticks_position('top')
cax.xaxis.set_label_position('top')
cb.set_label(r' $\chi$', labelpad=4, fontsize=14)
ax.set_xlabel(r"$\log_{10}(\mathrm{T/K})$")
ax.set_ylabel(r"$\log_{10}(\mathrm{n/cm^{-3}})$")
plt.show()

## Figure 5 — entropy change: continuous compression vs. no compression
Each grid is 200 × 200 and needs one ODE solve per point, so this step is **slow**. The grids are saved to `data/` and reused if they already exist. Columns are `T, N, Z`, the same format as the original `.dat` files, so you can drop those in under these names instead.

In [ ]:
metallicity = 0.3
period = 3*10**(16)
n_range = 10** np.linspace(-6, 0, 200)
T_range = 10** np.linspace(4, 7, 200)
T , N= np.meshgrid(T_range, n_range)

grids = {}
for kai in (1, 4, 10):     # kai = 1 is the no-compression case
    fname = f"data/entropy_kai{kai}_3e16_200.dat"
    if not os.path.exists(fname):
        Z_data1 = np.zeros_like(T, dtype=float)
        for i in range(T.shape[0]):
            for j in range(T.shape[1]):
                Z_data1[i, j] = entropy_diff(N[i, j], T[i, j], kai, metallicity, period) # log10(K_start / K_end)
        np.savetxt(fname, np.column_stack((T.ravel(), N.ravel(), Z_data1.ravel())), header="T, N, Z_data1", fmt="%.10e")
    data_k = np.loadtxt(fname, skiprows=1)
    grids[kai] = data_k[:, 2].reshape(len(n_range), len(T_range))

In [ ]:
"""
For 2D plot with dimensions of T and n, the specific values of n and T to give the same cooling time (1e17, 1e16, 3e16)
"""
T_range_c = np.linspace(4, 7, 5000)
n_range_c = np.linspace(-6, 0, 5000)
t_H = 1e17
log_tH = np.log10(t_H)
pairs_n = []
for ti in T_range_c:
    tcool_vals = tcool_P(10**ti, 10**n_range_c, metallicity)
    mask = (
        (tcool_vals - log_tH >= np.log10(0.999)) &
        (tcool_vals - log_tH < np.log10(1.001))
    )
    n_col = n_range_c[mask]
    if n_col.size > 0:
        pairs_n.extend([(ti, ni) for ni in n_col])
T_n_H = [p[0] for p in pairs_n]
n_H = [p[1] for p in pairs_n]
bottom_func = interp1d(T_n_H, n_H, kind = 'cubic', fill_value="extrapolate")

t_range = np.array((1e16, 3e16))
T_range_c = np.linspace(4, 7, 2000)
data = {}
for ti in t_range:
    data[ti] = []
    for Ti in T_range_c:
        def tcool_midel(x):
          return tcool_P(10**Ti, 10**x, metallicity) - np.log10(ti)
        if tcool_midel(bottom_func(Ti))*tcool_midel(0) <= 0:
          solution = root_scalar(tcool_midel, bracket = [bottom_func(Ti), 0], method='brentq')
          data[ti].append((Ti, solution.root))
T_16, n_16 = zip(*data[1e16])
T_3e16, n_3e16 = zip(*data[3e16])

In [ ]:
"""
Top panels: entropy change with compression for kai = 4 and 10
"""
for kai in (4, 10):
    Z_data1 = grids[kai]
    plt.figure()
    plt.imshow(
        Z_data1,
        extent=[np.log10(T_range.min()), np.log10(T_range.max()), np.log10(n_range.min()), np.log10(n_range.max())],
        origin='lower',
        aspect='auto',
        cmap='Reds',
        vmin= 0,
        vmax= 2
    )
    plt.colorbar(label=r"$log10(\frac{K_{rad}}{K_0})$")
    plt.xlabel(r"$log_{10}(T/\mathrm{K})$")
    plt.ylabel(r"$log_{10}(n/\mathrm{cm^{-3}})$")
    plt.title(rf"entropy of $\chi = {kai}$")
    plt.plot(T_n_H, n_H, "black", linestyle = "--", label = r"$t_{cool} = 10^{17}$")
    plt.plot(T_16, n_16, "black", linestyle = "-." , label = r"$t_{cool} = 10^{16}$")
    plt.plot(T_3e16, n_3e16, "black", linestyle = "-" , label = r"$t_{cool} = 10^{16.5}$")
    plt.legend(fontsize=12, loc = 'lower right')
    plt.show()

In [ ]:
"""
Bottom panels: entropy ratio of no compression and compression
"""
Z_data2 = grids[1]
for kai in (10, 4):
    Z_data1 = grids[kai]
    Entropy_ratio = Z_data1 - Z_data2
    fig, ax = plt.subplots()
    ax.xaxis.set_minor_locator(MultipleLocator(0.2))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))
    plt.imshow(
        Entropy_ratio,
        extent=[np.log10(T_range.min()), np.log10(T_range.max()), np.log10(n_range.min()), np.log10(n_range.max())],
        origin='lower',
        aspect='auto',
        cmap='seismic',
        vmin= -2,
        vmax= 2
    )
    plt.xlabel(r'$log_{10}(T/\mathrm{K})$')
    plt.ylabel(r'$log_{10}(n/\mathrm{cm^{-3}})$')
    plt.title(rf"entropy ratio of $\chi = 0$ and $\chi = {kai}$")
    plt.plot(T_n_H, n_H, "black", linestyle = "--")
    plt.plot(T_16, n_16, "black", linestyle = "-.")
    plt.plot(T_3e16, n_3e16, "black", linestyle = "-")
    plt.show()

fig, cax = plt.subplots(figsize=(.5, 7.5), constrained_layout=True)
norm = mpl.colors.Normalize(vmin=-2, vmax=2)
sm = mpl.cm.ScalarMappable(norm=norm, cmap='seismic')
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label(label=r"$log10(\frac{K_{\chi = 0}}{K_{\chi }})$")
plt.show()